In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('08_reporte')

# 08 - Reporte integral y consulta de linaje

Ejecutar despues del paso 7. Este notebook consulta la publicacion real, no usa ventas ficticias ni vuelve a cargar las etapas anteriores.

Muestra las ventas con sus cinco dimensiones, las auditorias de las etapas y el recorrido de un hecho hasta staging, RAW y origen. La consulta incluye los lotes y hashes de cabecera y detalle.

**Entrada:** `dw.fact_ventas_publicadas`, `auditoria.publicaciones` y tablas de linaje.

**Salida:** reportes CSV en `datos/reportes/`, auditoria SQL del paso 8 y su log JSONL. Se detiene si no hay una publicacion aprobada, faltan relaciones o existen rechazos posteriores.

In [ ]:
from conexion_sqlserver import BD_DESTINO

engine_destino = auditoria_etapa.engine
print(f'Reporte integral de: {BD_DESTINO}')

## Secuencia de verificacion

1. Exigir la ultima publicacion APROBADA.
2. Consultar la vista de ventas y sus dimensiones.
3. Verificar cobertura de linaje y ausencia de rechazos posteriores.
4. Mostrar auditorias y recorrer el linaje de una linea.
5. Exportar los reportes y registrar el linaje del archivo.

Para consultar una factura o linea especifica, cambiar `NUMERO_FACTURA` o `NUMERO_LINEA` en la celda de consulta. Con ambos en None se muestra el primer hecho publicado.

In [ ]:
import json
import pandas as pd
from sqlalchemy import text

with engine_destino.connect() as conn:
    publicacion = pd.read_sql_query('''
        SELECT * FROM auditoria.publicaciones
        WHERE id_auditoria = (SELECT MAX(id_auditoria) FROM auditoria.publicaciones)
    ''', conn)
if publicacion.empty or publicacion.iloc[0]['estado'] != 'APROBADO':
    raise RuntimeError('No hay una publicacion aprobada vigente; ejecutar y revisar el paso 7')
ID_PUBLICACION = publicacion.iloc[0]['id_publicacion']

with engine_destino.connect() as conn:
    reporte_ventas = pd.read_sql_query('''
        SELECT f.*, t.fecha, c.nombre_cliente, p.nombre_producto,
               v.nombre_completo AS vendedor, ci.ciudad, ci.estado_provincia, ci.pais
        FROM dw.fact_ventas_publicadas f
        JOIN dw.dim_tiempo t ON t.id_tiempo = f.id_tiempo
        JOIN dw.dim_cliente c ON c.id_cliente_sk = f.id_cliente_sk
        JOIN dw.dim_producto p ON p.id_producto_sk = f.id_producto_sk
        JOIN dw.dim_vendedor v ON v.id_vendedor_sk = f.id_vendedor_sk
        JOIN dw.dim_ciudad ci ON ci.id_ciudad_sk = f.id_ciudad_sk
        ORDER BY f.numero_factura, f.numero_linea
    ''', conn, coerce_float=False)
    cantidad_linaje = conn.execute(text('''
        SELECT COUNT(*) FROM auditoria.linaje_ventas WHERE id_publicacion = :id
    '''), {'id': ID_PUBLICACION}).scalar_one()
    rechazados_actuales = conn.execute(text('''
        SELECT COUNT(*) FROM auditoria.linaje_ventas l
        WHERE l.id_publicacion = :id AND (
            EXISTS (SELECT 1 FROM quarantine.invoice_lines q
                WHERE q.id_raw = l.id_raw_linea AND q.lote_id = l.lote_linea)
            OR EXISTS (SELECT 1 FROM quarantine.invoices q
                WHERE q.id_raw = l.id_raw_factura AND q.lote_id = l.lote_factura)
            OR EXISTS (SELECT 1 FROM raw.validacion_calidad v
                WHERE v.id_raw = l.id_raw_linea AND v.lote_id = l.lote_linea AND v.es_critico = 1)
            OR EXISTS (SELECT 1 FROM raw.validacion_calidad_invoices v
                WHERE v.id_raw = l.id_raw_factura AND v.lote_id = l.lote_factura AND v.es_critico = 1)
        )
    '''), {'id': ID_PUBLICACION}).scalar_one()
if reporte_ventas.empty or len(reporte_ventas) != cantidad_linaje or rechazados_actuales:
    raise RuntimeError('La publicacion cambio o tiene rechazos posteriores; volver a ejecutar el paso 7')
print(f'Publicacion vigente: {ID_PUBLICACION}; hechos: {len(reporte_ventas)}')
display(pd.DataFrame(json.loads(publicacion.iloc[0]['controles_json']).items(),
                     columns=['control', 'cumple']))
display(reporte_ventas.head(20))

In [ ]:
NUMERO_FACTURA = None
NUMERO_LINEA = None
seleccion = reporte_ventas
if NUMERO_FACTURA is not None:
    seleccion = seleccion.loc[seleccion['numero_factura'].eq(NUMERO_FACTURA)]
if NUMERO_LINEA is not None:
    seleccion = seleccion.loc[seleccion['numero_linea'].eq(NUMERO_LINEA)]
if seleccion.empty:
    raise ValueError('La factura o linea solicitada no existe en la publicacion')
ID_HECHO = str(int(seleccion.iloc[0]['id_fact_venta_sk']))

sql_camino = """
WITH relaciones AS (
    SELECT DISTINCT l.tabla_origen, l.clave_origen, l.tabla_destino, l.clave_destino,
                    l.id_ejecucion, e.etapa
    FROM auditoria.linaje_registros l
    JOIN auditoria.ejecuciones_etapa e ON e.id_ejecucion = l.id_ejecucion
    WHERE e.estado = 'OK'
), camino AS (
    SELECT r.*, 1 AS profundidad FROM relaciones r
    WHERE r.tabla_destino = 'dw.fact_ventas' AND r.clave_destino = :id
    UNION ALL
    SELECT r.*, c.profundidad + 1
    FROM relaciones r
    JOIN camino c ON r.tabla_destino = c.tabla_origen AND r.clave_destino = c.clave_origen
    WHERE c.profundidad < 8
)
SELECT DISTINCT * FROM camino ORDER BY profundidad, etapa, tabla_origen
"""
with engine_destino.connect() as conn:
    auditorias = pd.read_sql_query('''
        SELECT id_ejecucion, etapa, tabla, estado, inicio, fin, entradas, salidas, relaciones
        FROM auditoria.ejecuciones_etapa ORDER BY inicio DESC
    ''', conn)
    camino_linaje = pd.read_sql_query(text(sql_camino), conn, params={'id': ID_HECHO})
    linaje_raw = pd.read_sql_query(text('''
        SELECT * FROM auditoria.linaje_ventas
        WHERE id_publicacion = :publicacion AND id_fact_venta_sk = :hecho
    '''), conn, params={'publicacion': ID_PUBLICACION, 'hecho': int(ID_HECHO)})
display(auditorias.head(30))
print(f'Linaje por etapa del hecho {ID_HECHO}:')
display(camino_linaje)
print('Registros RAW de cabecera y detalle, con lote y hash:')
display(linaje_raw)

In [ ]:
DIRECTORIO_REPORTES = carpeta_pipeline / 'datos' / 'reportes'
DIRECTORIO_REPORTES.mkdir(parents=True, exist_ok=True)
archivo_reporte = DIRECTORIO_REPORTES / f'ventas_{auditoria_etapa.id_ejecucion}.csv'
reporte_ventas.to_csv(archivo_reporte, index=False, encoding='utf-8')
archivo_linaje = DIRECTORIO_REPORTES / f'linaje_{auditoria_etapa.id_ejecucion}.csv'
camino_linaje.to_csv(archivo_linaje, index=False, encoding='utf-8')
resumen_ventas = reporte_ventas.groupby('fecha')[['cantidad', 'monto_neto', 'monto_impuesto']].sum()
display(resumen_ventas)
print(f'Reporte: {archivo_reporte}')
print(f'Linaje: {archivo_linaje}')

## Evidencia generada

El reporte muestra datos reales publicados y permite vincular un hecho con sus cabeceras, detalles, dimensiones, registros RAW, lotes, hashes y ejecuciones de auditoria.

Los CSV se crean al ejecutar esta notebook. Los logs de cada etapa quedan en `logs/`; las relaciones y estados se conservan en SQL Server. Los archivos pueden contener informacion comercial y nombres: aplicar los permisos y la retencion acordados en gobierno.

Este paso no corrige RAW, no vacia cuarentena y no modifica hechos de negocio.

## Cierre

Verificar que cada etapa tenga auditoria OK, que cada hecho tenga linaje completo y que los reportes correspondan a la ultima publicacion aprobada. Una etapa EN_CURSO indica que no se ejecuto su cierre; ERROR exige revisar el log y revalidar antes de publicar.

In [ ]:
auditoria_etapa.finalizar(globals())